In [ ]:
!pip install imageio rasterio pandas pillow

In [5]:
from pathlib import Path
import shutil
import math
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
import imageio.v2 as imageio
from PIL import Image, ImageDraw

# -----------------------------
# CONFIG
# -----------------------------
s2_tiff_dir = Path(r"dataset_out/sent2_tiff")      # Optical GeoTIFFs
s1_tiff_dir = Path(r"dataset_out/sent1_tiff")      # SAR GeoTIFFs
csv_path    = Path(r"d:/antigravity/sat/combined_pairs_with_bbox.csv") 

clip = (0.0, 0.3)     # reflectance range mapped to 0..255 for Optical
max_size = 500        # Smaller preview size for gallery performance

out_dir = s2_tiff_dir / "_preview_gallery_pairs"
out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# HELPERS
# -----------------------------
def safe_name(s: str) -> str:
    s = str(s).strip().replace("\\", "_").replace("/", "_")
    keep = []
    for ch in s:
        if ch.isalnum() or ch in "._-":
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)[:220]

def to_reflectance(arr: np.ndarray) -> np.ndarray:
    a = arr.astype(np.float32)
    mx = np.nanmax(a) if np.isfinite(a).any() else 0.0
    if mx > 1.5:
        a = a / 10000.0
    return a

def clip_to_uint8(arr: np.ndarray, clip=(0.0, 0.3), nodata=None) -> np.ndarray:
    lo, hi = clip
    a = to_reflectance(arr)
    mask = np.isfinite(a)
    if nodata is not None:
        mask &= (a != nodata)
    out = np.zeros_like(a, dtype=np.uint8)
    if np.any(mask):
        x = np.clip(a, lo, hi)
        x = (x - lo) / (hi - lo + 1e-12)
        x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
        out = (x * 255.0).round().astype(np.uint8)
    return out

def read_downsampled(src: rasterio.io.DatasetReader, max_size: int):
    h, w = src.height, src.width
    scale = max(h, w) / float(max_size)
    if scale <= 1:
        return src.read(), 1.0
    out_h = max(1, int(h / scale))
    out_w = max(1, int(w / scale))
    data = src.read(out_shape=(src.count, out_h, out_w), resampling=Resampling.bilinear)
    actual_scale_w = w / out_w
    return data, actual_scale_w

def get_poly_coords(row: pd.Series, transform, scale: float = 1.0):
    corners = ["ul", "ur", "br", "bl"]
    pixels = []
    if "obj_ul_lon" not in row:
        return None
    try:
        for c in corners:
            lon = float(row[f"obj_{c}_lon"])
            lat = float(row[f"obj_{c}_lat"])
            
            # Check for NaNs to avoid RuntimeWarning in rasterio
            if math.isnan(lon) or math.isnan(lat):
                return None
                
            r, c_idx = rasterio.transform.rowcol(transform, lon, lat)
            x = c_idx / scale
            y = r / scale
            pixels.append((x, y))
    except (ValueError, KeyError, TypeError):
        return None
    if pixels:
        pixels.append(pixels[0])
    return pixels

def draw_poly_on_rgb(rgb_arr: np.ndarray, poly_coords, color=(255, 0, 0)) -> np.ndarray:
    if not poly_coords:
        return rgb_arr
    if rgb_arr.dtype != np.uint8:
        rgb_arr = rgb_arr.astype(np.uint8)
    pil_img = Image.fromarray(rgb_arr)
    draw = ImageDraw.Draw(pil_img)
    draw.line(poly_coords, fill=color, width=3)
    return np.array(pil_img)

def normalize_sar_log(band: np.ndarray, nodata=None) -> np.ndarray:
    b = band.astype(np.float32)
    mask = np.isfinite(b)
    if nodata is not None:
        mask &= (b != nodata)
    if not np.any(mask):
        return np.zeros_like(b, dtype=np.uint8)
    valid_data = np.maximum(b[mask], 0)
    log_data = np.log1p(valid_data)
    p2, p98 = np.percentile(log_data, (2, 98))
    if p98 - p2 < 1e-6:
        return np.zeros_like(b, dtype=np.uint8)
    b_log = np.log1p(np.maximum(b, 0))
    norm = (b_log - p2) / (p98 - p2)
    norm = np.clip(norm, 0, 1)
    norm = np.nan_to_num(norm, nan=0.0)
    return (norm * 255).round().astype(np.uint8)

def find_optical_tiff_for_row(row, s2_tiff_dir: Path) -> Path | None:
    s2_file = str(row.get("s2_file", ""))
    base = Path(s2_file).name
    stem = Path(base).stem
    for ext in [".tif", ".tiff", ".TIF", ".TIFF"]:
        p = s2_tiff_dir / (stem + ext)
        if p.exists(): return p
    candidates = sorted(list(s2_tiff_dir.glob(stem + "*.tif")) + list(s2_tiff_dir.glob(stem + "*.tiff")))
    return candidates[0] if candidates else None

def find_sar_tiff_for_row(row, s1_tiff_dir: Path) -> Path | None:
    s1_file = str(row.get("s1_file", ""))
    base = Path(s1_file).name
    stem = Path(base).stem
    for ext in [".tif", ".tiff", ".TIF", ".TIFF"]:
        p = s1_tiff_dir / (stem + ext)
        if p.exists(): return p
    candidates = sorted(list(s1_tiff_dir.glob(stem + "*.tif")) + list(s1_tiff_dir.glob(stem + "*.tiff")))
    if candidates: return candidates[0]
    patch_name = str(row.get("patch_name", "")).strip()
    if patch_name:
        hits = sorted(s1_tiff_dir.glob(f"*{patch_name}*.tif*"))
        if hits:
            return hits[0]
    return None

def extract_class_code(row) -> str:
    tag = str(row.get("label_tag", "")).lower()
    code = tag.split("-")[0] if "-" in tag else tag[:2]
    if code in {"ow", "oc", "nw", "nc"}:
        return code
    return "unknown"

def guess_rgb_indices(src):
    desc = list(src.descriptions) if src.descriptions else []
    def f(keys): 
        dlow = [(x or "").lower() for x in desc]
        for k in keys:
            for i, n in enumerate(dlow): 
                if k in n: return i
        return None
    r = f(["b04", "band_4", "band4", "red"])
    g = f(["b03", "band_3", "band3", "green"])
    b = f(["b02", "band_2", "band2", "blue"])
    if r is not None and g is not None and b is not None:
        return (r, g, b)
    if src.count >= 4:
        return (3, 2, 1)
    return None

# -----------------------------
# MAIN LOOP
# -----------------------------
if not csv_path.exists():
    print(f"CSV not found: {csv_path}")
else:
    df = pd.read_csv(csv_path)
    items = []
    print(f"Processing all {len(df)} rows...")
    
    count = 0
    for idx, row in df.iterrows():
        if count % 50 == 0:
            print(f"Processed {count} images...")
        count += 1
        
        sar_tif = find_sar_tiff_for_row(row, s1_tiff_dir)
        s2_tif  = find_optical_tiff_for_row(row, s2_tiff_dir)
        cls = extract_class_code(row)
        patch_name = row.get("patch_name", f"sample_{idx:03d}")
        sample_id = safe_name(patch_name)
        
        sar_out_name = None
        rgb_out_name = None
        file_note = []
        
        # --- PROCESS SAR ---
        if sar_tif and sar_tif.exists():
            sar_out_name = f"{sample_id}__SAR.png"
            try:
                with rasterio.open(sar_tif) as src:
                    data, scale = read_downsampled(src, max_size)
                    band1 = data[0]
                    sar_u8 = normalize_sar_log(band1, nodata=src.nodata)
                    sar_rgb = np.stack([sar_u8]*3, axis=-1)
                    poly = get_poly_coords(row, src.transform, scale)
                    sar_draw = draw_poly_on_rgb(sar_rgb, poly, color=(255, 0, 0))
                    imageio.imwrite((out_dir / sar_out_name).as_posix(), sar_draw)
            except Exception as e:
                # print(f"Error SAR: {e}")
                sar_out_name = None
                file_note.append("SAR Error")
        else:
            file_note.append("SAR Missing")

        # --- PROCESS OPTICAL ---
        if s2_tif and s2_tif.exists():
            rgb_out_name = f"{sample_id}__RGB.png"
            try:
                with rasterio.open(s2_tif) as src:
                    data, scale = read_downsampled(src, max_size)
                    poly = get_poly_coords(row, src.transform, scale)
                    rgb_idx = guess_rgb_indices(src)
                    if rgb_idx and max(rgb_idx) < src.count:
                        r, g, b = rgb_idx
                        R = clip_to_uint8(data[r], clip, src.nodata)
                        G = clip_to_uint8(data[g], clip, src.nodata)
                        B = clip_to_uint8(data[b], clip, src.nodata)
                        rgb = np.stack([R,G,B], axis=-1)
                        rgb_draw = draw_poly_on_rgb(rgb, poly, color=(255, 0, 0))
                        imageio.imwrite((out_dir / rgb_out_name).as_posix(), rgb_draw)
                    else:
                        rgb_out_name = None
                        file_note.append("RGB Indices Fail")
            except Exception as e:
                # print(f"Error Optical: {e}")
                rgb_out_name = None
                file_note.append("Optical Error")
        else:
            file_note.append("Optical Missing")

        items.append({
            "sample_id": sample_id,
            "patch_name": patch_name,
            "class": cls,
            "sar_png": sar_out_name,
            "rgb_png": rgb_out_name,
            "note": "; ".join(file_note),
        })
        
    # --- GENERATE HTML ---
    html = []
    html_top = """<!doctype html>
<html>
<head>
<meta charset="utf-8" />
<title>Dataset Selector</title>
<style>
  body { font-family: sans-serif; margin: 0; background: #f9f9f9; }
  .top-bar { 
    position: sticky; top: 0; background: #fff; 
    z-index: 1000; padding: 15px 30px; border-bottom: 2px solid #ddd; 
    box-shadow: 0 4px 10px rgba(0,0,0,0.1);
    display: flex; justify-content: space-between; align-items: center;
  }
  .top-bar h2 { margin: 0; font-size: 1.5em; color: #333; }
  .controls { display: flex; gap: 15px; align-items: center; }
  .btn { 
    background: #007bff; color: white; border: none; padding: 10px 20px; 
    font-size: 16px; border-radius: 6px; cursor: pointer; 
    transition: background 0.2s;
    font-weight: bold;
  }
  .btn:hover { background: #0056b3; }
  .btn:active { transform: translateY(1px); }
  .grid { 
    display: grid; 
    grid-template-columns: repeat(auto-fill, minmax(360px, 1fr)); 
    gap: 25px; 
    padding: 30px;
    max-width: 1600px;
    margin: 0 auto;
  }
  .card { 
    background: #fff;
    border: 3px solid transparent;
    border-radius: 12px; 
    padding: 15px; 
    cursor: pointer; 
    transition: all 0.2s;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
    position: relative;
    user-select: none;
  }
  .card:hover { transform: translateY(-3px); box-shadow: 0 6px 15px rgba(0,0,0,0.15); }
  
  /* SELECTION STATE */
  .card.selected { 
    border-color: #007bff; 
    background-color: #f0f8ff; 
    box-shadow: 0 0 15px rgba(0,123,255,0.4); 
  }
  .card.selected::after {
    content: '✓';
    position: absolute;
    top: 10px;
    right: 10px;
    background: #007bff;
    color: white;
    width: 25px;
    height: 25px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: bold;
  }

  .card-header { font-weight: bold; margin-bottom: 10px; display: flex; justify-content: space-between; align-items: center; }
  .tag { background: #444; color: #fff; padding: 3px 8px; border-radius: 4px; font-size: 0.85em; }
  .imgs { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }
  img { width: 100%; border-radius: 8px; object-fit: cover; aspect-ratio: 1/1; background: #eee; }
  .img-label { font-size: 0.8em; text-align: center; color: #666; margin-top: 5px; }
  .note { color: #d9534f; font-size: 0.8em; margin-bottom: 5px; font-weight: bold; }
</style>
<script>
  function toggleSelection(el) {
    el.classList.toggle('selected');
    updateCount();
  }
  
  function updateCount() {
    const count = document.querySelectorAll('.card.selected').length;
    document.getElementById('sel-count').innerText = count;
  }
  
  function copySelected() {
    const selected = document.querySelectorAll('.card.selected');
    const ids = Array.from(selected).map(el => el.getAttribute('data-id'));
    
    if (ids.length === 0) {
        alert('No images selected!');
        return;
    }
    
    const text = ids.join('\n');
    
    // Fallback for secure context issues
    if (navigator.clipboard && navigator.clipboard.writeText) {
        navigator.clipboard.writeText(text).then(() => {
            alert(`Copied ${ids.length} IDs to clipboard!`);
        }).catch(err => {
            fallbackCopy(text);
        });
    } else {
        fallbackCopy(text);
    }
  }
  
  function fallbackCopy(text) {
      const textArea = document.createElement("textarea");
      textArea.value = text;
      textArea.style.position = "fixed";  // Avoid scrolling to bottom
      document.body.appendChild(textArea);
      textArea.focus();
      textArea.select();
      try {
          const successful = document.execCommand('copy');
          const msg = successful ? 'successful' : 'unsuccessful';
          if(successful) alert('Copied IDs to clipboard (fallback)!');
          else alert('Copy failed.');
      } catch (err) {
          console.error('Fallback copy error', err);
          alert('Copy failed completely. Check console.');
      }
      document.body.removeChild(textArea);
  }
</script>
</head>
<body>
<div class="top-bar">
  <div class="title-block">
      <h2>Dataset Selector</h2>
  </div>
  <div class="controls">
      <span>Selected: <b id="sel-count" style="font-size: 1.2em">0</b></span>
      <button class="btn" onclick="copySelected()">Copy Selected IDs</button>
  </div>
</div>

<div class="grid">
"""
    html.append(html_top)
    
    for it in items:
        pid = it['patch_name']
        cls = it['class']
        note = f"<div class='note'>{it['note']}</div>" if it['note'] else ""
        
        # The onclick is on the wrapper div
        html.append(f"<div class='card' onclick='toggleSelection(this)' data-id='{pid}'>")
        
        html.append(f"<div class='card-header'><span>{pid}</span> <span class='tag'>{cls}</span></div>")
        if note: html.append(note)
        
        html.append("<div class='imgs'>")
        
        # SAR
        html.append("<div>")
        if it['sar_png']:
            html.append(f"<img src='{it['sar_png']}' title='SAR'>")
        else:
            html.append("<div style='background:#eee; aspect-ratio:1/1; display:flex; align-items:center; justify-content:center; border-radius:8px;'>No SAR</div>")
        html.append("<div class='img-label'>SAR</div></div>")
            
        # Optical
        html.append("<div>")
        if it['rgb_png']:
            html.append(f"<img src='{it['rgb_png']}' title='Optical'>")
        else:
            html.append("<div style='background:#eee; aspect-ratio:1/1; display:flex; align-items:center; justify-content:center; border-radius:8px;'>No Optical</div>")
        html.append("<div class='img-label'>Optical RGB</div></div>")
            
        html.append("</div></div>")
    
    html.append("</div></body></html>")
    
    (out_dir / "gallery.html").write_text("\n".join(html), encoding="utf-8")
    print("Gallery created at:", out_dir / "gallery.html")


Processing all 521 rows...
Processed 0 images...
Processed 50 images...
Processed 100 images...
Processed 150 images...
Processed 200 images...
Processed 250 images...
Processed 300 images...
Processed 350 images...
Processed 400 images...
Processed 450 images...
Processed 500 images...
Gallery created at: dataset_out\sent2_tiff\_preview_gallery_pairs\gallery.html
